# MatchFormer Epipolar Fine-tuning — Google Colab

Fine-tunes MatchFormer Lite-LA with epipolar supervision on ScanNet.
Checkpoints are saved to **Google Drive** so training survives Colab disconnects.

**Run order (fresh session):** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7  
**Resume after disconnect:** Cell 1 → 3 → 4 → 7 (skip 2, 5, 6)

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Root of your ScanNet data on Drive
DATA_ROOT = '/content/drive/MyDrive/final_proj/data/scans'

# Checkpoints saved here — change _run3, _run4 etc. to avoid overwriting previous runs
DRIVE_CKPT_DIR = '/content/drive/MyDrive/final_proj/matchformer_checkpoints_run2'

os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f'Data root:      {DATA_ROOT}')
print(f'Checkpoint dir: {DRIVE_CKPT_DIR}')

## Cell 2 — Clone Repo & Install Dependencies
*(Skip on resume — repo and packages persist for the session)*

In [ ]:
REPO_URL = 'https://github.com/sid-raj-uc/match-former.git'
REPO_DIR = '/content/final_proj'

if not os.path.exists(REPO_DIR):
    print('Cloning repo...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest')
    !cd {REPO_DIR} && git pull

%cd /content/final_proj/MatchFormer

!pip install -q pytorch-lightning einops kornia timm loguru yacs
!pip install -q opencv-python-headless
print('Done.')

## Cell 3 — Verify Scenes on Drive

In [ ]:
import glob

%cd /content/final_proj/MatchFormer

scenes = sorted([d for d in os.listdir(DATA_ROOT)
                 if os.path.isdir(os.path.join(DATA_ROOT, d, 'exported', 'color'))])
print(f'Found {len(scenes)} scenes:')
total = 0
for s in scenes:
    n = len(glob.glob(os.path.join(DATA_ROOT, s, 'exported', 'color', '*.jpg')))
    total += n
    print(f'  {s}: {n} frames')
print(f'\nTotal frames: {total}')

## Cell 4 — Copy Data to Local SSD

Copies ScanNet from Drive to Colab's local SSD (`/content/`).  
Drive has high per-file latency that starves the dataloader — local SSD is ~5x faster.  
Takes ~5–10 min once per session; skipped automatically if already done.

In [ ]:
import os, time, glob, shutil, threading, subprocess

LOCAL_DATA  = '/content/scannet_data'
DRIVE_ZIPS  = '/content/drive/MyDrive/final_proj/scannet_zips'
EXPECTED_FRAMES = 23871

os.makedirs(LOCAL_DATA, exist_ok=True)

scenes = sorted([d for d in os.listdir(DATA_ROOT)
                 if os.path.isdir(os.path.join(DATA_ROOT, d))])

def copy_scene(scene):
    src       = os.path.join(DATA_ROOT, scene)
    dst       = os.path.join(LOCAL_DATA, scene)
    zip_drive = os.path.join(DRIVE_ZIPS, f'{scene}.zip')

    n_local = len(glob.glob(os.path.join(dst, 'exported/color/*.jpg')))
    n_drive = len(glob.glob(os.path.join(src, 'exported/color/*.jpg')))

    if n_local >= n_drive > 0:
        print(f'  [{scene}] Already local ({n_local} frames) — skipping')
        return

    # ── Fast path: zip exists on Drive (~30s per scene) ──
    if os.path.exists(zip_drive):
        size_gb = os.path.getsize(zip_drive) / 1e9
        print(f'  [{scene}] Zip found ({size_gb:.2f} GB) — copying + unzipping...')
        zip_local = f'/content/{scene}.zip'
        t0 = time.time()
        shutil.copy2(zip_drive, zip_local)
        subprocess.run(['unzip', '-q', zip_local, '-d', LOCAL_DATA], check=True)
        os.remove(zip_local)
        print(f'  [{scene}] Done in {time.time()-t0:.0f}s')

    # ── Slow path: no zip, copy file by file (~11 min per scene) ──
    else:
        print(f'  [{scene}] No zip found — copying {n_drive} frames directly (slow)...')
        t0 = time.time()
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'  [{scene}] Done in {(time.time()-t0)/60:.1f}m')

# ── Copy first scene synchronously so training can start immediately ──
print('=== Copying scene0000_00 first ===')
copy_scene(scenes[0])
print('Ready to train!\n')

# ── Copy remaining scenes in background while training runs ──
def copy_remaining(remaining):
    print(f'[Background] Copying {len(remaining)} remaining scenes...')
    for scene in remaining:
        copy_scene(scene)
    total = len(glob.glob(os.path.join(LOCAL_DATA, '*/exported/color/*.jpg')))
    print(f'\n[Background] All done! {total}/{EXPECTED_FRAMES} frames local.')
    print('[Background] Restart training cell to train on all scenes.')

if len(scenes) > 1:
    bg = threading.Thread(target=copy_remaining, args=(scenes[1:],), daemon=True)
    bg.start()
    print(f'[Background] Copying remaining {len(scenes)-1} scenes...')

print(f'\nTraining will read from: {LOCAL_DATA}')

## Cell 4b — Create Drive Zips for Fast Restarts
Run this **once** after all scenes are fully copied locally (while training is running is fine).  
Creates one zip per scene on Drive — future restarts skip the slow file-by-file copy and take ~5 min total instead of 2 hours.

In [ ]:
import os, time, glob, shutil, subprocess, threading

DRIVE_ZIPS = '/content/drive/MyDrive/final_proj/scannet_zips'
os.makedirs(DRIVE_ZIPS, exist_ok=True)

# Wait for background copy from cell 4 to finish before zipping
print('Waiting for all scenes to be fully copied locally...')
while True:
    total_frames = len(glob.glob(os.path.join(LOCAL_DATA, '*/exported/color/*.jpg')))
    print(f'  {total_frames}/{EXPECTED_FRAMES} frames local', end='\r')
    if total_frames >= EXPECTED_FRAMES:
        break
    time.sleep(30)
print(f'\nAll {total_frames} frames local — starting zip + upload in background...')

scenes = sorted([d for d in os.listdir(LOCAL_DATA)
                 if os.path.isdir(os.path.join(LOCAL_DATA, d))])

def create_zips():
    for scene in scenes:
        zip_drive = os.path.join(DRIVE_ZIPS, f'{scene}.zip')
        if os.path.exists(zip_drive):
            print(f'  [{scene}] Zip already on Drive — skipping')
            continue

        zip_local = f'/content/{scene}.zip'
        print(f'  [{scene}] Zipping from local SSD...')
        t0 = time.time()
        subprocess.run(['zip', '-0', '-r', '-q', zip_local, scene],
                       cwd=LOCAL_DATA, check=True)
        size_gb = os.path.getsize(zip_local) / 1e9
        print(f'  [{scene}] Zipped {size_gb:.2f} GB in {time.time()-t0:.0f}s — uploading...')

        t0 = time.time()
        shutil.copy2(zip_local, zip_drive)
        os.remove(zip_local)
        print(f'  [{scene}] Uploaded in {time.time()-t0:.0f}s')

    print('\nAll zips on Drive — future restarts will take ~5 min total.')

zip_thread = threading.Thread(target=create_zips, daemon=True)
zip_thread.start()
print('Zipping + uploading in background — training is not affected.')

## Cell 5 — Download Pretrained Weights
*(Skip on resume — weights persist for the session)*

In [ ]:
WEIGHTS_PATH  = 'model/weights/indoor-lite-LA.ckpt'
DRIVE_WEIGHTS = '/content/drive/MyDrive/final_proj/MatchFormer/model/weights/indoor-lite-LA.ckpt'

os.makedirs('model/weights', exist_ok=True)

if os.path.exists(WEIGHTS_PATH):
    print('Weights already present')
elif os.path.exists(DRIVE_WEIGHTS):
    shutil.copy(DRIVE_WEIGHTS, WEIGHTS_PATH)
    print('Copied weights from Drive')
else:
    print('WARNING: weights not found. Upload indoor-lite-LA.ckpt to Drive at:')
    print(' ', DRIVE_WEIGHTS)

## Cell 6 — Verify GPU & Sanity Check

Runs 50 steps on 5 pairs to confirm the pipeline works end-to-end.  
Loss should decrease. Takes ~1 min on L4.

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — no GPU!"}')
print(f'CUDA: {torch.cuda.is_available()}')

# Sanity check: 5 pairs, 50 steps, reads from Drive (fine for 5 pairs)
!python train_finetune.py \
    --overfit \
    --data_dir {DATA_ROOT} \
    --checkpoint_dir {DRIVE_CKPT_DIR}/overfit \
    --batch 4 \
    --steps 50

print('Sanity check done — loss should be decreasing!')

## Cell 7 — Full Fine-Tuning

**Auto-resume is on** — if Colab disconnects, re-run Cell 1 → 3 → 4 → this cell.  
It will pick up from the last checkpoint in `DRIVE_CKPT_DIR` automatically.

Key settings:
- `--data_dir` reads from **local SSD** (fast)
- `--precision bf16` — ~2x speedup on L4 via native bf16 tensor cores
- `--tau 50.0` — soft epipolar mask, better for multi-scene diversity
- `--save_every 500` — saves to Drive every 500 steps (20 checkpoints total)

In [ ]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!python train_finetune.py \
    --data_dir {LOCAL_DATA} \
    --checkpoint_dir {DRIVE_CKPT_DIR} \
    --steps 10000 \
    --tau 50.0 \
    --batch 4 \
    --workers 4 \
    --lr 1e-4 \
    --save_every 500 \
    --precision bf16

# --- Resume command (same as above, auto-resume is built in) ---
# !python train_finetune.py \
#     --data_dir {LOCAL_DATA} \
#     --checkpoint_dir {DRIVE_CKPT_DIR} \
#     --steps 10000 \
#     --tau 50.0 --batch 4 --workers 4 --save_every 500 --precision bf16

## Cell 8 — Benchmark After Training

In [ ]:
TRAINED_CKPT = f'{DRIVE_CKPT_DIR}/last.ckpt'
LOCAL_CKPT   = 'model/weights/epipolar-finetuned.ckpt'

shutil.copy(TRAINED_CKPT, LOCAL_CKPT)
print(f'Checkpoint copied to {LOCAL_CKPT}')

!python run_benchmark.py \
    --num_pairs 100 \
    2>&1 | tee benchmark_finetuned_log.txt

shutil.copy('benchmark_finetuned_log.txt', f'{DRIVE_CKPT_DIR}/benchmark_finetuned_log.txt')
print('Results saved to Drive.')